In [1]:
from __future__ import print_function
import os
from glob import glob
import numpy as np
# from pixmappy.pixmappy import DESMaps, Gnomonic, Tweak, DECamTweak
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates.angles import Angle
from astropy import units as u
from astrometry.util.fits import *
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from random import randint
# from legacypipe.py.legacypipe.survey import radec_at_mjd
from scipy.optimize import leastsq, minimize, curve_fit
from scipy.interpolate import RectBivariateSpline
import sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('py_files'), '..')))
from py_files import create_brick_catalogue
from py_files import brick_ringmaps
from py_files import brick_DCR
from py_files import brick_lateralmaps

%load_ext autoreload
%autoreload 2

In [2]:
gaia_dir = '/pscratch/sd/d/dstn/forced-motions-dr10/gaia-stars/'
rmaps_fname = '/pscratch/sd/n/nelfalou/rm-gaia-stars/ringmaps.fits'
lmaps_fname = '/pscratch/sd/n/nelfalou/lm-gaia-stars/lateralmaps.fits'

In [3]:
crpixes = {'S1': (2151.2, 14826.0),
 'S2': (2151.2, 10566.67),
 'S3': (2151.2, 6307.333),
 'N1': (-103.2001, 14826.0),
 'N2': (-103.2001, 10566.67),
 'N3': (-103.2001, 6307.333),
 'S8': (4405.6, 12696.33),
 'S9': (4405.6, 8437.0),
 'S14': (6660.0, 12696.33),
 'S15': (6660.0, 8437.0),
 'S20': (8914.4, 10566.67),
 'S25': (11168.8, 8437.0),
 'N8': (-2357.6, 12696.33),
 'N9': (-2357.6, 8437.0),
 'N14': (-4612.0, 12696.33),
 'N15': (-4612.0, 8437.0),
 'N20': (-6866.4, 10566.67),
 'N25': (-9120.8, 8437.0),
 'S10': (4405.6, 4177.667),
 'S11': (4405.6, -81.66665),
 'S12': (4405.6, -4341.0),
 'S13': (4405.6, -8600.334),
 'S18': (6660.0, -4341.0),
 'S19': (6660.0, -8600.334),
 'S16': (6660.0, 4177.667),
 'S17': (6660.0, -81.66665),
 'S21': (8914.4, 6307.333),
 'S22': (8914.4, 2048.0),
 'S23': (8914.4, -2211.333),
 'S24': (8914.4, -6470.667),
 'S26': (11168.8, 4177.667),
 'S27': (11168.8, -81.66665),
 'S28': (11168.8, -4341.0),
 'S29': (13423.2, 6307.333),
 'S30': (13423.2, 2048.0),
 'S31': (13423.2, -2211.333),
 'N4': (-103.2001, 2048.0),
 'N5': (-103.2001, -2211.333),
 'N6': (-103.2001, -6470.667),
 'N7': (-103.2001, -10730.0),
 'S4': (2151.2, 2048.0),
 'S5': (2151.2, -2211.333),
 'S6': (2151.2, -6470.667),
 'S7': (2151.2, -10730.0),
 'N10': (-2357.6, 4177.667),
 'N11': (-2357.6, -81.66665),
 'N12': (-2357.6, -4341.0),
 'N13': (-2357.6, -8600.334),
 'N18': (-4612.0, -4341.0),
 'N19': (-4612.0, -8600.334),
 'N16': (-4612.0, 4177.667),
 'N17': (-4612.0, -81.66665),
 'N21': (-6866.4, 6307.333),
 'N22': (-6866.4, 2048.0),
 'N23': (-6866.4, -2211.333),
 'N24': (-6866.4, -6470.667),
 'N26': (-9120.8, 4177.667),
 'N27': (-9120.8, -81.66665),
 'N28': (-9120.8, -4341.0),
 'N29': (-11375.2, 6307.333),
 'N30': (-11375.2, 2048.0),
 'N31': (-11375.2, -2211.333),
 'FS1': (11168.8, -8600.334),
 'FS2': (13423.2, -6470.667),
 'FS3': (15677.6, -81.6666),
 'FS4': (15677.6, 2129.667),
 'FN1': (-9120.8, -8600.334),
 'FN2': (-11375.2, -6470.667),
 'FN3': (-13629.6, -81.6666),
 'FN4': (-13629.6, 2129.667)}

In [4]:
fns = glob(gaia_dir + 'gaia-forced-*.fits')
fns.sort()
len(fns)

45

In [ ]:
FF, TT = [], []
for fn in fns:
    forced = fits_table(fn)
    tfn = gaia_dir + 'gaia-tractor-' + fn.split('-')[-1]
    tractor = fits_table(tfn)
    
    FF.append(forced)
    TT.append(tractor)
    
F = merge_tables(FF)
T = merge_tables(TT)
del FF, TT
len(forced)

In [ ]:
objmap = dict([((bid,oid),i) for i,(bid,oid) in enumerate(zip(T.brickid, T.objid))])
F.t_index = np.array([objmap.get((bid,oid), -1) for bid,oid in zip(F.brickid, F.objid)])
F.sigma_rd = np.sqrt(1. / F.full_fit_dra_ivar + 1. / F.full_fit_ddec_ivar)
F.ref_cat = T.ref_cat[F.t_index]
F.pmra = T.pmra[F.t_index]

F.gaia_g = T.gaia_phot_g_mean_mag[F.t_index]
F.gaia_bp = T.gaia_phot_bp_mean_mag[F.t_index]
F.gaia_rp = T.gaia_phot_rp_mean_mag[F.t_index]
F.flux_g = T.flux_g[F.t_index]
# F.flux_r = T.flux_r[F.t_index]
F.flux_i = T.flux_i[F.t_index]
# F.flux_z = T.flux_z[F.t_index]
T.color = -2.5 * (np.log10(T.flux_g / T.flux_i))
F.color = T.color[F.t_index]

p = np.array([crpixes[c] for c in F.ccdname])
F.fpx = F.rm_full_fit_x - p[:,0]
F.fpy = F.rm_full_fit_y - p[:,1]

ux = F.fpx / np.hypot(F.fpx, F.fpy)
uy = F.fpy / np.hypot(F.fpx, F.fpy)
F.dr = (F.rm_full_fit_x - F.x) * ux + (F.rm_full_fit_y - F.y) * uy

In [ ]:
ccd = 'N4'
J = np.flatnonzero((F.ccdname == ccd)
                            * np.isin(F.filter, 'z')
                            * (F.full_fit_dra_ivar > 1e4)
                            * (F.full_fit_dra != 0.)
                            * (F.dqmask == 0))

plt.figure(figsize=(14,12))
# ex = (0,2048,0,4096)
plt.subplot(1,2,1)
plt.hexbin(F.x[J], F.y[J], C=F.full_fit_x[J] - F.x[J], reduce_C_function=np.median, gridsize=(150,400),
           vmin=-0.05, vmax=+0.05, extent=ex);
plt.axis(ex);
plt.subplot(1,2,2)
plt.hexbin(F.x[J], F.y[J], C=F.full_fit_y[J] - F.y[J], reduce_C_function=np.median, gridsize=(150,400),
           vmin=-0.05, vmax=+0.05, extent=ex);
plt.axis(ex);